In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import rs1090

import pandas as pd
from rtlsdr import RtlSdr
import numpy as np

import plotly
from plotly.graph_objects import Scatter
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
init_notebook_mode(connected=False)
import plotly.express as px

import cosmosdr

import cosmosdr.signal_acquisition as s_acq
import cosmosdr.signal_processing as s_proc
from cosmosdr.plotting import create_base_figure

try:
    sdr.close()
except:
    pass

In [ ]:
center_freq = 1090e6
sample_rate = 1e6
n_reads = 512
n_samples = 1024  # more samples seems to just make it noisy

# Start up the SDR connection
try:
    sdr.close()
except:
    pass

sdr = s_acq.get_sdr(center_freq=center_freq, sample_rate=sample_rate)

s = s_acq.acquire_signal(sdr, n_reads=n_reads, n_samples=n_samples)

s_fft = np.zeros([n_reads, n_samples], dtype=np.float64)

# Perform fft on each reading, store simply the magnitude for plotting
for i in range(n_reads):
    s_i_freqs, s_i_fft, s_i_mag_fft, s_i_angle_fft = s_proc.get_frequency_space_np(s[i, :], center_freq=center_freq, sample_rate=sample_rate)
    s_fft[i, :] = s_i_mag_fft

# Calculate the highest signal peak from each read, post FFT (basically, it will be high if a signal was received at that time)
signal_peaks = s_fft.max(axis=1)
# grab the read with the highest peak
received_signal = s_fft[signal_peaks.argmax()]

### Plot the peakiest peak from the 100 reads
fig = create_base_figure()
scatter_trace = Scatter(y=received_signal)  # todo x labels etc.
fig.add_trace(scatter_trace)




In [ ]:
# Show a spectrogram

n = 256

plot_window_1 = int(s_fft.shape[1] / 2) - n
plot_window_2 = int(s_fft.shape[1] / 2) + n

# "x":f"{int(center_freq/1000/1000)} MHz"
x_plot=np.round(s_i_freqs[plot_window_1:plot_window_2],1)  / 1000 / 1000
y_plot = s_fft[:, plot_window_1:plot_window_2]

fig = px.imshow(
    # ignore the frequency either size of the target (@1000)
    y_plot,
    x=x_plot,
    labels={"x":"Frequency(MHz)", "y":"time (upper is earlier)", "color":"signal magnitude"},
    width=1024,
    height=800,
    aspect="auto"
)
fig.show()

# Speed test, how much time is spent just instantiating the sdr?
(a lot)

In [ ]:
%%time
for i in range(5):
    print(i)
    get_signal()


In [ ]:
sdr = get_sdr()

In [ ]:
%%time

for i in range(5):
    print(i)
    sdr.read_samples(4096)

sdr.close()


# Initial plots

In [ ]:
s = get_signal()

In [ ]:
s.shape

In [ ]:
df = pd.DataFrame(s, columns=["signal"])

# Split the signal's complex numbers into real and imaginary components
df["signal_real"] = df["signal"].apply(lambda row: row.real)
df["signal_imag"] = df["signal"].apply(lambda row: row.imag)

print(df.shape)
df.head()

In [ ]:
df_plot = df.iloc[:,1:]
fig = px.scatter(df_plot, y="signal_imag", color="signal_real")
fig.show()

# FFT
https://pysdr.org/content/frequency_domain.html#fft-in-python

### Simple intro

In [ ]:
# time
t = np.arange(1000)
# signal
s = np.cos(t/10)
px.line(s)

In [ ]:
# signal in frequency domain
S = np.fft.fft(s)

In [ ]:
# Interpret the complex numbers as magnitude and phase
S_mag = np.abs(S)
S_phase = np.angle(S)

In [ ]:
px.line(S_mag)

### 'shift' the fft signal, because the it is formatted in a conveninent way in the numpy output
The default output is [0, -ve freqs, +ve freqs], which is dumb, we want it zero centered

In [ ]:
S = np.fft.fftshift(S)
# Interpret the complex numbers as magnitude and phase
S_mag = np.abs(S)
S_phase = np.angle(S)

In [ ]:
# So we see, apparently there are two signals... ?
px.line(S_mag)

# FFT on a signal

In [ ]:
def plot_frequency_space(s, center_freq, sample_rate):
    # The output x axis ranges from -(x/2) to (x/2), where x is the sample rate, and 0 is centered on the target frequency
    # In this case, it is 102.2 MHz +/- ~2 MHz
    
    # fft the input, and center it on 0
    s_fft = np.fft.fftshift(np.fft.fft(s))
    
    dfft = pd.DataFrame(s_fft, columns=["signal_fft"])
    
    # Split the signal's complex numbers into real and imaginary components
    dfft["signal_real_fft"] = dfft["signal_fft"].apply(lambda row: row.real)
    dfft["signal_imag_fft"] = dfft["signal_fft"].apply(lambda row: row.imag)
    
    # Calculate magnitude and angle of the complex signal
    dfft["signal_mag_fft"] = dfft["signal_fft"].apply(np.abs)
    dfft["signal_angle_fft"] = dfft["signal_fft"].apply(np.angle)
    
    # Estimate frequencies 
    dfft["frequency"] = np.linspace(center_freq-(sample_rate/2), center_freq+(sample_rate/2), dfft.shape[0])
    
    fig = px.scatter(dfft, x="frequency", y="signal_mag_fft", color="signal_angle_fft")
    fig.show()

In [ ]:
center_freq=102.7e6
sample_rate=1e6

s = get_signal(center_freq=center_freq, sample_rate=sample_rate)
plot_frequency_space(center_freq=99.7e6)

In [ ]:
# # ...
# sdr.sample_rate = 2.4e5 # Hz
# # ...

# fft_size = 512
# num_rows = 500
# x = sdr.read_samples(2048) # get rid of initial empty samples
# x = sdr.read_samples(fft_size*num_rows) # get all the samples we need for the spectrogram
# spectrogram = np.zeros((num_rows, fft_size))
# for i in range(num_rows):
#     spectrogram[i,:] = 10*np.log10(np.abs(np.fft.fftshift(np.fft.fft(x[i*fft_size:(i+1)*fft_size])))**2)
# extent = [(sdr.center_freq + sdr.sample_rate/-2)/1e6,
#             (sdr.center_freq + sdr.sample_rate/2)/1e6,
#             len(x)/sdr.sample_rate, 0]
# plt.imshow(spectrogram, aspect='auto', extent=extent)
# plt.xlabel("Frequency [MHz]")
# plt.ylabel("Time [s]")
# plt.show()